<a href="https://colab.research.google.com/github/the-name-is-madhu/AI-lab/blob/main/Resolution_using_FOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import re
import heapq

# --- Parser for User Input ---
def parse_literal(lit_str):
    """Parses '~food(x)' or 'likes(John, x)' into (is_negated, pred, args)."""
    lit_str = lit_str.strip()
    is_negated = False
    if lit_str.startswith(("~", "!", "¬")):
        is_negated = True
        lit_str = lit_str.lstrip("~!¬").strip()

    match = re.match(r"^([A-Za-z0-9_]+)\(([^)]*)\)$", lit_str)
    if not match:
        raise ValueError(f"Invalid literal format: '{lit_str}'. Expected: Predicate(arg1, arg2)")

    pred = match.group(1)
    args = tuple(arg.strip() for arg in match.group(2).split(",") if arg.strip())
    return (is_negated, pred, args)

def parse_clause(clause_str):
    """Parses '~food(x) | likes(John, x)' into a frozenset of literals."""
    literals = clause_str.split("|")
    return frozenset(parse_literal(lit) for lit in literals if lit.strip())

def parse_kb(kb_str):
    """Splits a single string containing all clauses separated by ';', '&', or newlines."""
    raw_clauses = re.split(r"[;&\n]+", kb_str)
    kb = [parse_clause(c) for c in raw_clauses if c.strip()]
    if not kb:
        raise ValueError("Knowledge Base cannot be empty.")
    return kb

# --- FOL Unification & Resolution Engine ---
def is_variable(term):
    return isinstance(term, str) and len(term) > 0 and term.islower()

def occur_check(var, term, subst):
    if var == term:
        return True
    if is_variable(term) and term in subst:
        return occur_check(var, subst.get(term), subst)
    return False

def unify_var(var, x, subst):
    if var in subst:
        return unify(subst.get(var), x, subst)
    if is_variable(x) and x in subst:
        return unify(var, subst.get(x), subst)
    if occur_check(var, x, subst):
        return None
    new_subst = subst.copy()
    new_subst[var] = x
    return new_subst

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if x == y:
        return subst
    if is_variable(x):
        return unify_var(x, y, subst)
    if is_variable(y):
        return unify_var(y, x, subst)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        if not x:
            return subst
        head_x, *tail_x = x
        head_y, *tail_y = y
        first_unify = unify(head_x, head_y, subst)
        if first_unify is None:
            return None
        return unify(tuple(tail_x), tuple(tail_y), first_unify)
    return None

def apply_subst(literal, subst):
    neg, pred, args = literal
    resolved_args = []
    for arg in args:
        while is_variable(arg) and arg in subst:
            arg = subst.get(arg)
        resolved_args.append(arg)
    return (neg, pred, tuple(resolved_args))

def standardize_variables(clause, counter):
    mapping = {}
    new_literals = set()
    for neg, pred, args in clause:
        new_args = []
        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = f"{arg}_{counter}"
                    counter += 1
                new_args.append(mapping.get(arg))
            else:
                new_args.append(arg)
        new_literals.add((neg, pred, tuple(new_args)))
    return frozenset(new_literals), counter

def format_clause(clause):
    if not clause:
        return "NIL (Empty Clause □)"
    parts = []
    for neg, pred, args in sorted(clause):
        prefix = "~" if neg else ""
        parts.append(f"{prefix}{pred}({', '.join(args)})")
    return " | ".join(parts)

def resolve(ci, cj):
    resolvents = []
    for li in ci:
        neg_i, pred_i, args_i = li
        for lj in cj:
            neg_j, pred_j, args_j = lj
            if pred_i == pred_j and neg_i != neg_j:
                subst = unify(args_i, args_j, {})
                if subst is not None:
                    new_ci = {apply_subst(l, subst) for l in ci if l != li}
                    new_cj = {apply_subst(l, subst) for l in cj if l != lj}
                    resolvents.append((frozenset(new_ci | new_cj), subst))
    return resolvents

def print_minimal_proof(empty_clause, parents):
    """Backtracks from NIL to extract and print only the steps used in the proof."""
    proof_steps = []
    visited = set()

    def trace(clause):
        if clause in visited or clause not in parents:
            return
        visited.add(clause)
        c1, c2, subst = parents.get(clause)
        trace(c1)
        trace(c2)
        proof_steps.append((c1, c2, subst, clause))

    trace(empty_clause)

    print(f"\n=== Minimized Proof ({len(proof_steps)} Steps) ===")
    for step_num, (c1, c2, subst, res) in enumerate(proof_steps, 1):
        print(f"Step {step_num}: [{format_clause(c1)}]  +  [{format_clause(c2)}]")
        print(f"        Subst: {subst}  =>  {format_clause(res)}")
    print("\nSUCCESS: Empty clause derived! Query is PROVED TRUE.")

def fol_resolution_minimized(kb, query):
    counter = 1
    kb_clauses = []
    seen = set()

    for c in kb:
        std_c, counter = standardize_variables(c, counter)
        if std_c not in seen:
            kb_clauses.append(std_c)
            seen.add(std_c)

    # Negated query forms the initial Set of Support (SOS)
    q_neg, q_pred, q_args = query
    negated_query = frozenset({(not q_neg, q_pred, q_args)})
    std_q, counter = standardize_variables(negated_query, counter)
    seen.add(std_q)

    print("\n=== Standardized KB & Negated Query ===")
    for idx, c in enumerate(kb_clauses, 1):
        print(f"C{idx}: {format_clause(c)}")
    print(f"Negated Query (SOS): {format_clause(std_q)}")

    # Priority queue for Unit Preference: (clause_length, tie_breaker, clause)
    tie_breaker = 0
    sos_queue = [(len(std_q), tie_breaker, std_q)]
    sos_list = [std_q]
    parents = {}

    while sos_queue:
        _, _, current = heapq.heappop(sos_queue)

        # Resolve current SOS clause against all KB clauses and previous SOS clauses
        candidates = kb_clauses + sos_list
        for other in candidates:
            if current == other:
                continue
            for resolvent, subst in resolve(current, other):
                if resolvent not in seen:
                    seen.add(resolvent)
                    parents[resolvent] = (current, other, subst)
                    if len(resolvent) == 0:
                        print_minimal_proof(resolvent, parents)
                        return True
                    tie_breaker += 1
                    sos_list.append(resolvent)
                    heapq.heappush(sos_queue, (len(resolvent), tie_breaker, resolvent))

    print("\nFAILED: Empty clause cannot be derived. Query CANNOT be proved.")
    return False

# --- Interactive Main Loop ---
if __name__ == "__main__":

    print("Syntax: Separate clauses with ';', use '|' for OR, '~' for NOT\n")

    try:
        raw_kb = input("Enter all KB clauses      : ")
        kb = parse_kb(raw_kb)

        raw_query = input("Enter query to prove: ")
        query = parse_literal(raw_query)

        fol_resolution_minimized(kb, query)

    except Exception as e:
        print(f"\nInput Error: {e}")


Syntax: Separate clauses with ';', use '|' for OR, '~' for NOT

Enter all KB clauses: ~food(x) | likes(John, x); food(Apple); food(Vegetables); ~eats(x, y) | killed(x) | food(y); eats(Anil, Peanuts); alive(Anil); ~eats(Anil, x) | eats(Harry, x); ~alive(x) | ~killed(x); killed(x) | alive(x)
Enter query to prove: likes(John, Peanuts)

=== Standardized KB & Negated Query ===
C1: likes(John, x_1) | ~food(x_1)
C2: food(Apple)
C3: food(Vegetables)
C4: food(y_2) | killed(x_3) | ~eats(x_3, y_2)
C5: eats(Anil, Peanuts)
C6: alive(Anil)
C7: eats(Harry, x_4) | ~eats(Anil, x_4)
C8: ~alive(x_5) | ~killed(x_5)
C9: alive(x_6) | killed(x_6)
Negated Query (SOS): ~likes(John, Peanuts)

=== Minimized Proof (5 Steps) ===
Step 1: [~likes(John, Peanuts)]  +  [likes(John, x_1) | ~food(x_1)]
        Subst: {'x_1': 'Peanuts'}  =>  ~food(Peanuts)
Step 2: [~food(Peanuts)]  +  [food(y_2) | killed(x_3) | ~eats(x_3, y_2)]
        Subst: {'y_2': 'Peanuts'}  =>  killed(x_3) | ~eats(x_3, Peanuts)
Step 3: [killed(x_3) |